# Lens — Day 1: Core Loop

**Goal:** laptop webcam → Moondream scene description → Claude Haiku directive → spoken aloud

Run each cell in order. By the end, Lens will watch your webcam and tell you what shot to get next.

## Before you start
1. Copy `.env.example` to `.env` and fill in your API keys
2. Install deps: `pip install -r requirements.txt` (run once in terminal)
3. Get a free Gemini key at https://aistudio.google.com (needed for Day 2)

In [ ]:
# Setup — run from the lens/ directory
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())  # searches upward from CWD — works from any subdirectory

print('ANTHROPIC_API_KEY:', 'SET' if os.environ.get('ANTHROPIC_API_KEY') else 'MISSING')
print('GOOGLE_API_KEY:', 'SET' if os.environ.get('GOOGLE_API_KEY') else 'MISSING (needed Day 2)')

## Step 1: Capture a frame from your webcam

In [ ]:
from camera import capture_frame
from IPython.display import display

frame = capture_frame()
display(frame)
print(f'Frame captured: {frame.size}')

## Step 2: Moondream describes the scene

First run downloads the model (~1.2GB). Subsequent runs use the cached model.

In [ ]:
from agents.environment_monitor import describe_scene

scene = describe_scene(frame)
print('Scene description:')
print(scene)

## Step 3: Director decides what to capture next

In [ ]:
from agents.director import get_directive
from manifest import STUB_MANIFEST
from memory import CoverageMemory

mem = CoverageMemory('../data/coverage_memory.json')

directive = get_directive(
    manifest=STUB_MANIFEST,
    scene=scene,
    memory_summary=mem.summary(),
)

import json
print(json.dumps(directive, indent=2))

## Step 4: Speak the directive aloud

In [ ]:
from tts import speak

if directive.get('speak') and directive.get('directive'):
    print(f"Speaking: {directive['directive']}")
    speak(directive['directive'])
else:
    print('No directive to speak (scene is fine).')

## Step 5: Continuous loop (runs until you stop it with Kernel → Interrupt)

This is the real Lens loop. Every 10 seconds it:
1. Captures a frame
2. Gets scene description from Moondream
3. Calls Director only if scene changed
4. Speaks directive if needed (with 90s cooldown)

In [ ]:
import time
from agents.environment_monitor import describe_scene, scene_changed
from agents.director import get_directive
from camera import capture_frame
from manifest import STUB_MANIFEST
from memory import CoverageMemory
from tts import speak

CAPTURE_INTERVAL = int(os.environ.get('CAPTURE_INTERVAL', 10))
DIRECTIVE_COOLDOWN = int(os.environ.get('DIRECTIVE_COOLDOWN', 90))

mem = CoverageMemory('../data/coverage_memory.json')
last_scene = ''
cycle = 0

print('Lens is watching. Press Kernel → Interrupt to stop.\n')

while True:
    cycle += 1
    print(f'--- Cycle {cycle} ---')

    frame = capture_frame()
    scene = describe_scene(frame)
    mem.update_scene(scene)
    print(f'Scene: {scene[:80]}...')

    if not scene_changed(scene, last_scene):
        print('Scene unchanged — skipping Director call')
    elif mem.seconds_since_last_directive() < DIRECTIVE_COOLDOWN:
        secs = int(DIRECTIVE_COOLDOWN - mem.seconds_since_last_directive())
        print(f'Cooldown active — {secs}s until next directive')
    else:
        directive = get_directive(
            manifest=STUB_MANIFEST,
            scene=scene,
            memory_summary=mem.summary(),
            last_directive=mem.state['last_directive'],
        )
        print(f'Directive: {directive["directive"]} [{directive["severity"]}]')

        if directive.get('speak') and directive.get('directive'):
            speak(directive['directive'])
            mem.record_directive(directive['directive'])

    last_scene = scene
    print(f'Waiting {CAPTURE_INTERVAL}s...\n')
    time.sleep(CAPTURE_INTERVAL)